# Fraud Detection Model Training Pipeline

**Model:** `FRAUD_DETECTION_MODEL` (XGBClassifier)  
**Schema:** `DEMO_DEV.FRAUD_INTELLIGENCE`  
**Purpose:** Binary classification — predict whether a transaction is fraudulent.

## Label Sourcing Strategy
1. **Primary:** `GLD_ML_TRAINING_SET.TARGET_LABEL`
2. **Secondary (feedback loop):** `BRZ_RAW_ALERTS` dispositions (`DISPOSITION='CONFIRMED_FRAUD'` or `SAR_FILED=TRUE`)
3. If no positive labels exist, a warning is logged but training proceeds.

> **Note:** The Streamlit app does not yet write analyst decisions back to tables. Once the feedback write-back is implemented, this notebook will automatically pick up fraud labels on retraining.

In [ ]:
!pip install xgboost

In [ ]:
import warnings
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("fraud_model_training")

session = get_active_session()
session.sql("USE ROLE AGENT_DEMO_ROLE").collect()
session.sql("USE SCHEMA DEMO_DEV.FRAUD_INTELLIGENCE").collect()

MODEL_NAME = "FRAUD_DETECTION_MODEL"
VERSION_NAME = f"V{datetime.now().strftime('%Y%m%d_%H%M%S')}"
MIN_POSITIVE_LABELS_THRESHOLD = 10

print(f"Session active. Model version: {VERSION_NAME}")

In [ ]:
%%sql -r training_data
SELECT
    TRANSACTION_ID,
    CUSTOMER_ID,
    TRANSACTION_AMOUNT,
    TRANSACTION_TYPE,
    MERCHANT_CATEGORY,
    CHANNEL,
    ACCOUNT_AGE_DAYS,
    CREDIT_LIMIT,
    CURRENT_BALANCE,
    ACCOUNT_TYPE,
    UTILIZATION_RATIO,
    RISK_SEGMENT,
    KYC_COMPLETE,
    ADDRESS_CHANGE_COUNT,
    IS_SYNTHETIC_FLAG,
    IS_MERGER_DUP_FLAG,
    CUSTOMER_TENURE_DAYS,
    PRIOR_ALERT_COUNT_CUSTOMER,
    ALERT_WITHIN_7D,
    TXN_COUNT_1H,
    TXN_COUNT_24H,
    TXN_AMOUNT_1H,
    TXN_AMOUNT_24H,
    DISTINCT_MERCHANTS_24H,
    TARGET_LABEL
FROM DEMO_DEV.FRAUD_INTELLIGENCE.GLD_ML_TRAINING_SET

In [ ]:
%%sql -r feedback_labels
SELECT
    a.TRANSACTION_ID,
    a.CUSTOMER_ID,
    a.DISPOSITION,
    a.SAR_FILED,
    a.ALERT_STATUS
FROM DEMO_DEV.FRAUD_INTELLIGENCE.BRZ_RAW_ALERTS a
WHERE a.DISPOSITION IS NOT NULL
   OR a.SAR_FILED = TRUE

In [ ]:
df_train = training_data.copy()
df_feedback = feedback_labels.copy()

has_feedback = len(df_feedback) > 0

if has_feedback:
    fraud_txns = df_feedback[
        (df_feedback["DISPOSITION"].str.upper() == "CONFIRMED_FRAUD") |
        (df_feedback["SAR_FILED"] == True)
    ]["TRANSACTION_ID"].unique()

    df_train.loc[
        df_train["TRANSACTION_ID"].isin(fraud_txns), "TARGET_LABEL"
    ] = 1
    logger.info(f"Feedback loop active: {len(fraud_txns)} fraud-labeled transactions merged.")
else:
    logger.warning(
        "No decision feedback found in BRZ_RAW_ALERTS. "
        "Using TARGET_LABEL from GLD_ML_TRAINING_SET as-is. "
        "Feedback write-back from Streamlit app is not yet implemented."
    )

positive_count = int(df_train["TARGET_LABEL"].sum())
negative_count = len(df_train) - positive_count

print(f"Label distribution — Fraud: {positive_count}, Non-Fraud: {negative_count}")

if positive_count < MIN_POSITIVE_LABELS_THRESHOLD:
    warnings.warn(
        f"Only {positive_count} positive labels found (threshold: {MIN_POSITIVE_LABELS_THRESHOLD}). "
        f"Model may not learn meaningful fraud patterns. Proceeding with training anyway.",
        UserWarning
    )

In [ ]:
CATEGORICAL_COLS = ["TRANSACTION_TYPE", "MERCHANT_CATEGORY", "CHANNEL", "ACCOUNT_TYPE", "RISK_SEGMENT"]
BOOLEAN_COLS = ["KYC_COMPLETE", "IS_SYNTHETIC_FLAG", "IS_MERGER_DUP_FLAG"]
NUMERIC_COLS = [
    "TRANSACTION_AMOUNT", "ACCOUNT_AGE_DAYS", "CREDIT_LIMIT", "CURRENT_BALANCE",
    "UTILIZATION_RATIO", "ADDRESS_CHANGE_COUNT", "CUSTOMER_TENURE_DAYS",
    "PRIOR_ALERT_COUNT_CUSTOMER", "ALERT_WITHIN_7D", "TXN_COUNT_1H",
    "TXN_COUNT_24H", "TXN_AMOUNT_1H", "TXN_AMOUNT_24H", "DISTINCT_MERCHANTS_24H"
]

EXPECTED_OHE_COLUMNS = [
    "TRANSACTION_TYPE_CASH_ADVANCE", "TRANSACTION_TYPE_PAYMENT",
    "TRANSACTION_TYPE_PURCHASE", "TRANSACTION_TYPE_TRANSFER",
    "MERCHANT_CATEGORY_CASH", "MERCHANT_CATEGORY_DINING", "MERCHANT_CATEGORY_GAS",
    "MERCHANT_CATEGORY_GROCERY", "MERCHANT_CATEGORY_HEALTHCARE",
    "MERCHANT_CATEGORY_ONLINE", "MERCHANT_CATEGORY_RETAIL", "MERCHANT_CATEGORY_TRAVEL",
    "CHANNEL_BRANCH", "CHANNEL_MOBILE", "CHANNEL_ONLINE",
    "ACCOUNT_TYPE_CREDIT", "ACCOUNT_TYPE_SAVINGS",
    "RISK_SEGMENT_LOW", "RISK_SEGMENT_MEDIUM"
]

df_encoded = pd.get_dummies(df_train, columns=CATEGORICAL_COLS, prefix_sep="_", dtype=float)

for col in BOOLEAN_COLS:
    df_encoded[col] = df_encoded[col].astype(float)

for col in EXPECTED_OHE_COLUMNS:
    if col not in df_encoded.columns:
        df_encoded[col] = 0.0

FEATURE_COLS = NUMERIC_COLS + BOOLEAN_COLS + EXPECTED_OHE_COLUMNS

for col in NUMERIC_COLS:
    df_encoded[col] = pd.to_numeric(df_encoded[col], errors="coerce").fillna(0.0)

X = df_encoded[FEATURE_COLS].astype(float)
y = df_encoded["TARGET_LABEL"].astype(int)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {FEATURE_COLS}")

In [ ]:
if positive_count > 0 and negative_count > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    logger.info(f"Stratified split: train={len(X_train)}, test={len(X_test)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    logger.warning("Single-class data — using random split (no stratification possible).")

print(f"Train set: {len(X_train)} rows | Test set: {len(X_test)} rows")
print(f"Train label dist: {y_train.value_counts().to_dict()}")
print(f"Test label dist: {y_test.value_counts().to_dict()}")

In [ ]:
scale_pos_weight = negative_count / max(positive_count, 1)

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

print(f"Model trained. scale_pos_weight={scale_pos_weight:.2f}")
print(f"Parameters: n_estimators={model.n_estimators}, max_depth={model.max_depth}")

In [ ]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

num_classes = len(np.unique(y_test))

if num_classes > 1:
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])

    print("=== Model Evaluation ===")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc_roc:.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")
else:
    precision, recall, f1, auc_roc = 0.0, 0.0, 0.0, 0.5
    logger.warning(
        "Single-class test set — evaluation metrics are not meaningful. "
        "Setting defaults: precision=0, recall=0, f1=0, auc_roc=0.5"
    )
    print(f"Single-class predictions. All predictions: {np.unique(y_pred)}")

metrics_dict = {
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1, 4),
    "auc_roc": round(auc_roc, 4),
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "positive_labels": positive_count,
    "negative_labels": negative_count,
}
print(f"\nMetrics dict: {metrics_dict}")

In [ ]:
%%sql -r prior_metrics
SELECT
    PRECISION_SCORE,
    RECALL_SCORE,
    F1_SCORE,
    AUC_ROC,
    MODEL_VERSION
FROM DEMO_DEV.FRAUD_INTELLIGENCE.SLV_MODEL_PERFORMANCE_LOG
WHERE MODEL_NAME = 'FRAUD_DETECTION_MODEL'
ORDER BY EVAL_DATE DESC
LIMIT 1

In [ ]:
register_model = True
comparison_notes = []

if len(prior_metrics) > 0:
    prior = prior_metrics.iloc[0]
    prior_f1 = float(prior["F1_SCORE"])
    prior_auc = float(prior["AUC_ROC"])
    prior_version = prior["MODEL_VERSION"]

    print(f"=== Comparison vs Prior ({prior_version}) ===")
    print(f"  F1:      {prior_f1:.4f} -> {f1:.4f} ({'improved' if f1 >= prior_f1 else 'degraded'})")
    print(f"  AUC-ROC: {prior_auc:.4f} -> {auc_roc:.4f} ({'improved' if auc_roc >= prior_auc else 'degraded'})")

    if f1 < prior_f1 and num_classes > 1:
        comparison_notes.append(f"WARNING: F1 degraded from {prior_f1} to {f1}.")
        logger.warning(f"F1 degraded from {prior_f1} to {f1}. Registering anyway.")
else:
    print("No prior model metrics found. This is the first training run.")
    comparison_notes.append("First training run — no prior metrics to compare.")

if num_classes <= 1:
    comparison_notes.append(
        "WARNING: Single-class training data. Model trained on non-fraud only. "
        "Metrics are not meaningful until feedback labels are available."
    )
    logger.warning("Single-class training — model registered as baseline.")

print(f"\nRegistering model: {register_model}")
print(f"Notes: {comparison_notes}")

In [ ]:
from snowflake.ml.model import task as ml_task

registry = Registry(session=session, database_name="DEMO_DEV", schema_name="FRAUD_INTELLIGENCE")

sample_input = X_test.head(100).fillna(0.0)

model_ref = registry.log_model(
    model=model,
    model_name=MODEL_NAME,
    version_name=VERSION_NAME,
    sample_input_data=sample_input,
    metrics=metrics_dict,
    target_platforms=["WAREHOUSE"],
    task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION,
    comment=f"Trained on {len(X_train)} rows. Positive labels: {positive_count}. Notes: {'; '.join(comparison_notes)}",
)

print(f"Model registered: {MODEL_NAME} / {VERSION_NAME}")
print(f"Target platform: WAREHOUSE")
print(f"Functions available: PREDICT, PREDICT_PROBA, EXPLAIN")

In [ ]:
%%sql -r insert_perf_log
INSERT INTO DEMO_DEV.FRAUD_INTELLIGENCE.SLV_MODEL_PERFORMANCE_LOG
    (EVAL_DATE, MODEL_NAME, MODEL_VERSION, PRECISION_SCORE, RECALL_SCORE, F1_SCORE, AUC_ROC, LOGGED_AT)
VALUES
    (CURRENT_DATE(), '{{MODEL_NAME}}', '{{VERSION_NAME}}', {{precision}}, {{recall}}, {{f1}}, {{auc_roc}}, CURRENT_TIMESTAMP())

In [ ]:
%%sql -r update_metadata
MERGE INTO DEMO_DEV.FRAUD_INTELLIGENCE.SLV_FRAUD_MODEL_METADATA AS tgt
USING (
    SELECT
        '{{MODEL_NAME}}' AS MODEL_NAME,
        '{{VERSION_NAME}}' AS MODEL_VERSION,
        'XGBClassifier' AS MODEL_TYPE,
        'ACTIVE' AS MODEL_STATUS,
        CURRENT_TIMESTAMP() AS LAST_TRAINED_AT,
        CURRENT_TIMESTAMP() AS CREATED_AT
) AS src
ON tgt.MODEL_NAME = src.MODEL_NAME
WHEN MATCHED THEN UPDATE SET
    tgt.MODEL_VERSION = src.MODEL_VERSION,
    tgt.MODEL_TYPE = src.MODEL_TYPE,
    tgt.MODEL_STATUS = src.MODEL_STATUS,
    tgt.LAST_TRAINED_AT = src.LAST_TRAINED_AT
WHEN NOT MATCHED THEN INSERT
    (MODEL_NAME, MODEL_VERSION, MODEL_TYPE, MODEL_STATUS, LAST_TRAINED_AT, CREATED_AT)
VALUES
    (src.MODEL_NAME, src.MODEL_VERSION, src.MODEL_TYPE, src.MODEL_STATUS, src.LAST_TRAINED_AT, src.CREATED_AT)

## Training Complete

### Summary
- **Model:** `FRAUD_DETECTION_MODEL`
- **Version:** See `VERSION_NAME` variable above
- **Features:** 36 (14 numeric + 3 boolean + 19 one-hot encoded)
- **Algorithm:** XGBClassifier with balanced class weights

### Next Steps
1. Implement analyst decision write-back in Streamlit app (DISPOSITION, SAR_FILED)
2. Re-run this notebook after feedback accumulates to retrain with labeled fraud data
3. Monitor `SLV_MODEL_PERFORMANCE_LOG` for metric trends across versions